<a href="https://colab.research.google.com/github/Mscient/30-Days-Of-Code/blob/master/ML_piplines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

In [ ]:
df=pd.read_csv('/content/train.csv')

In [5]:
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [12]:
df.drop(['Ticket','Fare','Cabin'],axis=1,inplace=True)

KeyError: "['Ticket', 'Fare', 'Cabin'] not found in axis"

In [11]:
X_Train,x_test,Y_train,y_test=train_test_split(df.drop(columns=['Survived']),df['Survived'],test_size=0.2,random_state=42)

In [26]:
X_Train

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Embarked
331,332,1,"Partner, Mr. Austen",male,45.5,0,0,S
733,734,2,"Berriman, Mr. William John",male,23.0,0,0,S
382,383,3,"Tikkanen, Mr. Juho",male,32.0,0,0,S
704,705,3,"Hansen, Mr. Henrik Juul",male,26.0,1,0,S
813,814,3,"Andersson, Miss. Ebba Iris Alfrida",female,6.0,4,2,S
...,...,...,...,...,...,...,...,...
106,107,3,"Salkjelsvik, Miss. Anna Kristine",female,21.0,0,0,S
270,271,1,"Cairns, Mr. Alexander",male,NaN,0,0,S
860,861,3,"Hansen, Mr. Claus Peter",male,41.0,2,0,S
435,436,1,"Carter, Miss. Lucile Polk",female,14.0,1,2,S


In [14]:
x_test


,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Embarked
709,710,3,"Moubarek, Master. Halim Gonios (""William George"")",male,NaN,1,1,C
439,440,2,"Kvillner, Mr. Johan Henrik Johannesson",male,31.0,0,0,S
840,841,3,"Alhomaki, Mr. Ilmari Rudolf",male,20.0,0,0,S
720,721,2,"Harper, Miss. Annie Jessie ""Nina""",female,6.0,0,1,S
39,40,3,"Nicola-Yarred, Miss. Jamila",female,14.0,1,0,C
...,...,...,...,...,...,...,...,...
433,434,3,"Kallio, Mr. Nikolai Erland",male,17.0,0,0,S
773,774,3,"Elias, Mr. Dibo",male,NaN,0,0,C
25,26,3,"Asplund, Mrs. Carl Oscar (Selma Augusta Emilia...",female,38.0,1,5,S
84,85,2,"Ilett, Miss. Bertha",female,17.0,0,0,S


In [15]:
df.isnull().sum()


,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Embarked,2


In [27]:
df_age=SimpleImputer()
df_embark=SimpleImputer(strategy='most_frequent')

x_Train_age=df_age.fit_transform(X_Train[['Age']])
x_Train_embark=df_embark.fit_transform(X_Train[['Embarked']])

x_test_age=df_age.fit_transform(x_test[['Age']])
x_test_embark=df_embark.fit_transform(x_test[['Embarked']])

In [28]:
x_Train_embark.shape

(712, 1)

In [33]:
# 1. Change 'sparse' to 'sparse_output'
ohe_sex = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_embarked = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# 2. Fit and transform using the correct variable names
X_train_sex = ohe_sex.fit_transform(X_Train[['Sex']])
X_train_embarked = ohe_embarked.fit_transform(X_Train[['Embarked']]) # Fixed variable name here

X_test_sex = ohe_sex.transform(x_test[['Sex']])
X_test_embarked = ohe_embarked.transform(x_test[['Embarked']])

In [48]:
x_rem = X_Train.drop(columns=['PassengerId', 'Name', 'Sex', 'Age', 'Embarked'])
x_test = x_test.drop(columns=['PassengerId', 'Name', 'Sex', 'Age', 'Embarked'], errors='ignore')

In [49]:
# Combining the training data pieces
X_train_transformed = np.concatenate((x_rem, x_Train_age, X_train_sex, X_train_embarked), axis=1)

# Combining the testing data pieces
X_test_transformed = np.concatenate((x_test, x_test_age, X_test_sex, X_test_embarked), axis=1)

In [50]:
clf = DecisionTreeClassifier()
clf.fit(X_train_transformed,Y_train)

DecisionTreeClassifier()

In [51]:
y_pred = clf.predict(X_test_transformed)
y_pred

array([0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1,
       0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1,
       0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0,
       1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0,
       0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1,
       0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0,
       0, 1, 0])

In [52]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.7653631284916201